# Typed Output and Dependency Injection

**What you will build:** agents that return **validated Python objects** instead of text, and tools that can reach real **dependencies** (a database, the current user, config) safely. These two features are PydanticAI's signature.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/13_typed_output_and_di.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once per notebook.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.2 (2026)

import os, getpass
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"   # any id from openrouter.ai/models
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## Typed output with `output_type`

In 0.2 you got structured output by hand: JSON mode, then Pydantic validation, in a helper. PydanticAI folds that into one argument — `output_type`. Pass a Pydantic model and `result.output` is a validated instance of it, retried automatically if the model's first attempt doesn't fit.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Ticket(BaseModel):
    category: Literal["billing", "technical", "account", "other"]
    urgency:  Literal["low", "medium", "high"]
    summary:  str = Field(description="One-line summary")

triage = Agent(model, output_type=Ticket, system_prompt="Classify the support ticket.")

result = await triage.run("The app crashes whenever I open reports. I need this fixed today.")
print(result.output)
print("typed field:", result.output.urgency, "->", type(result.output).__name__)

`result.output` is a real `Ticket`, so `result.output.urgency` is a checked value you can branch on — no JSON parsing, no validation code. Compare that to the whole helper you wrote in 0.2.

## Dependency injection: give tools real data

Real tools need context: *which* customer, *which* database, *which* API key. PydanticAI's `deps` mechanism passes that in safely — you declare a `deps_type`, hand data to `agent.run(..., deps=...)`, and tools read it from `ctx.deps` via `@agent.tool`. The model never sees the raw dependency; only your tool code does.

In [ ]:
from dataclasses import dataclass
from pydantic_ai import RunContext

@dataclass
class SupportDeps:
    customer_name: str
    plan: str
    tickets_open: int

agent = Agent(
    model,
    deps_type=SupportDeps,
    system_prompt="You are a support agent. Use tools to look up the customer's account before answering.",
)

@agent.tool
def get_account(ctx: RunContext[SupportDeps]) -> str:
    """Return the current customer's account details."""
    d = ctx.deps
    return f"Customer {d.customer_name} is on the {d.plan} plan with {d.tickets_open} open tickets."

deps = SupportDeps(customer_name="Alex", plan="Pro", tickets_open=2)
result = await agent.run("Can you remind me which plan I'm on and how many tickets I have open?", deps=deps)
print(result.output)

The agent called `get_account`, which read the injected `SupportDeps` from `ctx.deps`. In a real app that same tool would query your database. Dependency injection is how you connect an agent to *your* systems without leaking them into the prompt — and without hard-coding globals into tools.

::::{dropdown} 🔧 Under the hood: why this beats the 0.2 helper
:color: info

In 0.2 you wrote `extract(model_class, text, instructions)` — build a schema, call with `response_format`, validate, and you had no retry if validation failed. `output_type` does all of that *and* automatically re-prompts the model when its output doesn't satisfy the schema, up to a retry limit. Same idea, hardened.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Combine both.** Give the support agent an `output_type` of a `Reply(message: str, needs_human: bool)` model so its answer is structured *and* it uses the account tool.
2. **Add a second tool** that reads `ctx.deps` — e.g. `list_recent_charges()` returning fake data — and ask a question that needs both tools.
3. **Prove isolation.** Confirm the model can't see `deps` directly: only your `@agent.tool` functions can. Ask it to "print your dependencies" and watch it fail.
::::

**What's next:** in **1.4** — **memory**. The multi-turn conversation you built by hand in 0.6 becomes a single argument.